In [31]:
from dotenv import load_dotenv
load_dotenv()

# Core imports
import os
import re
import json
import pandas as pd
import requests
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

MAX_STEPS = 8

In [32]:
df = pd.read_csv("moviecsv.csv")

def _norm(text: str) -> str:
    return re.sub(r"[^a-z0-9]+", " ", str(text).lower()).strip()

title_lookup = {_norm(t): t for t in df["Title"].dropna().unique()}

def _mentioned_titles(text):
    norm_text = _norm(text)
    return [orig for norm, orig in title_lookup.items() if norm in norm_text]

In [33]:
reviews = {}
if os.path.exists("data"):
    for file in os.listdir("data"):
        if file.endswith(".txt"):
            with open(os.path.join("data", file), "r", encoding="utf-8") as f:
                reviews[file] = f.read()

In [34]:
def _chunk_text(text: str, chunk_size=700, overlap=120):
    text = re.sub(r"\s+", " ", text).strip()
    chunks = []
    i = 0
    while i < len(text):
        chunks.append(text[i:i+chunk_size])
        i += chunk_size - overlap
    return chunks

doc_index = []
for filename, content in reviews.items():
    chunks = _chunk_text(content)
    for idx, chunk in enumerate(chunks, 1):
        doc_index.append({
            "title": filename,
            "source": filename,
            "page": idx,
            "text": chunk
        })

if doc_index:
    vectorizer = TfidfVectorizer(stop_words="english")
    X = vectorizer.fit_transform([d["text"] for d in doc_index])

In [35]:
def search_docs(query, top_k=3):
    """
    LLM DESCRIPTION:
    This tool performs semantic search over unstructured movie reviews and plot summaries. 
    Use it to understand themes, character motivations, plot details, or the overall tone of a film. 
    Input: Natural language query string.
    Expected Output: Top-3 relevant text chunks with source filename and page number.
    """
    q_vec = vectorizer.transform([query])
    sim = cosine_similarity(q_vec, X).flatten()
    top_indices = sim.argsort()[-top_k:][::-1]
    
    results = []
    for idx in top_indices:
        res = doc_index[idx]
        results.append({"text": res['text'], "source": res['source'], "page": res['page']})
        
    return {
        "tool": "search_docs",
        "input": query,
        "row_count": len(results),
        "results": results
    }

def query_data(query_or_question):
    """
    LLM DESCRIPTION:
    This tool queries a structured financial table containing movie stats (Title, Year, Genre, Gross, etc.).
    Use it for specific numbers, box office rankings, budgets, or filtering by genre/year.
    Input: A natural-language question or filtering intent.
    Expected Output: A table or scalar value with column names and row count.
    """
    data = df.copy()
    q = query_or_question.lower()
    
    mentions = _mentioned_titles(query_or_question)
    if mentions:
        data = data[data["Title"].isin(mentions)]
        
    if "highest" in q or "top" in q or "gross" in q:
        data = data.sort_values("WorldwideGross_M$", ascending=False)
    
    res_list = data.head(5).to_dict(orient="records")
    return {
        "tool": "query_data",
        "input": query_or_question,
        "row_count": len(res_list),
        "results": res_list
    }

def web_search(query):
    """
    LLM DESCRIPTION:
    Search the live web for recent movie news, latest award wins, or information not in local docs.
    Input: A short search query string (under 10 words).
    Expected Output: Top-3 result snippets with URL and publication date.
    """
    api_key = os.getenv("TAVILY_API_KEY")
    if not api_key:
        return {"tool": "web_search", "error": "API key missing", "results": []}

    try:
        payload = {"api_key": api_key, "query": query, "max_results": 3}
        resp = requests.post("https://api.tavily.com/search", json=payload, timeout=15)
        data = resp.json()
        results = []
        for res in data.get("results", []):
            results.append({"snippet": res.get("content"), "url": res.get("url"), "date": res.get("published_date")})
        return {"tool": "web_search", "input": query, "row_count": len(results), "results": results}
    except:
        return {"tool": "web_search", "row_count": 0, "results": []}

In [36]:
def llm_call(prompt):
    api_key = os.getenv("OPENAI_API_KEY")
    if not api_key:
        return "Error: OPENAI_API_KEY missing"
    try:
        r = requests.post(
            "https://api.openai.com/v1/chat/completions",
            headers={"Authorization": f"Bearer {api_key}"},
            json={
                "model": "gpt-4o-mini",
                "messages": [{"role": "user", "content": prompt}],
                "temperature": 0
            }
        )
        return r.json()["choices"][0]["message"]["content"]
    except:
        return ""

In [37]:
def _decide_next_step(question, context_history, step_count):
    prompt = f"""
    Question: {question}
    Context: {json.dumps(context_history, indent=2)}
    Step: {step_count}/8
    
    Decide: Need tool or ready for final answer?
    Tools: search_docs, query_data, web_search
    Return ONLY JSON: {{"tool": "name", "input": "query"}} OR {{"final_answer": true}}
    """
    res = llm_call(prompt)
    try:
        return json.loads(re.search(r'\{.*\}', res, re.DOTALL).group())
    except:
        return {"final_answer": True}

def run_agent(question):
    context = []
    print(f"Question: {question}")
    
    for step in range(1, 9):
        decision = _decide_next_step(question, context, step-1)
        if decision.get("final_answer"):
            break
            
        tool = decision.get("tool")
        inp = decision.get("input")
        
        if tool == "search_docs": res = search_docs(inp)
        elif tool == "query_data": res = query_data(inp)
        elif tool == "web_search": res = web_search(inp)
        else: break
        
        context.append({"step": step, "tool": tool, "input": inp, "result": res["results"]})
        print(f"Step {step}: tool={tool} input='{inp}'")
        print(f"result={str(res['results'])[:100]}...")

    if len(context) >= 8 and not decision.get("final_answer"):
        return "Refusal: Maximum tool call limit (8) reached."

    final_answer = llm_call(f"Evidence: {json.dumps(context)}\nQuestion: {question}\nCompose answer with citations.")
    print(f"Final Answer: {final_answer}")
    print(f"Steps used: {len(context)} / 8 max")
    return final_answer

In [38]:
run_agent("Compare the box office performance of Titanic and Inception.")

Question: Compare the box office performance of Titanic and Inception.
Final Answer: 
Steps used: 0 / 8 max


''